<a href="https://colab.research.google.com/github/Aldiss1/MPPM_Aldi/blob/main/JS06/Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas Lab

1. Buatlah model SVM dengan menggunakan data `voice.csv dengan ketentuan,
   1. Split data dengan rasio 70:30 dan 80:20 untuk setiap model yang akan dibangun.
      1. Gunakan model dengan kernel linier.
      2. Gunakan model dengan kernel polynomial.
      3. Gunakan model dengan kernel RBF.
   2. Tabulasikan performansi setiap split dan kernel berdasarkan metrik akurasi.
2. Gunakan data pada praktikum 5 untuk membuat model klasifikasi siang dan malam menggunakan SVM dengan kernel RBF menggunakan fitur histrogram. Gunakan rasio 80:20. Anda dapat bereksperimen dengan hyperparameter tunning dari kernel RBF. Catat performansi akurasinya!

In [3]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/MeachineLearning'

import os
print('Isi folder:', os.listdir(BASE))

Mounted at /content/drive
Isi folder: ['voice.csv', 'images']


## Jawaban Soal 1 — SVM pada `voice.csv`

### Langkah 1 - Load Data

voice.csv berisi 3.168 sampel suara dengan 20 fitur akustik dan kolom label (male/female). File voice.csv dibaca dari folder Google Drive (lihat bagian Persiapan).

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv(BASE + '/voice.csv')
print('Ukuran data :', df.shape)
print('Missing value:', df.isna().sum().sum())
print(df['label'].value_counts())

Ukuran data : (3168, 21)
Missing value: 0
label
male      1584
female    1584
Name: count, dtype: int64


### Langkah 2 - Pisahkan Fitur dan Label

Label diubah menjadi angka (female = 0, male = 1).

In [5]:
from sklearn.preprocessing import LabelEncoder

X = df.drop(columns='label')
y = LabelEncoder().fit_transform(df['label'])

print('Jumlah fitur:', X.shape[1])
print('Kelas       :', np.unique(y))

Jumlah fitur: 20
Kelas       : [0 1]


### Langkah 3 - Model SVM Tiga Kernel dengan Dua Rasio Split

Untuk setiap rasio (70:30 dan 80:20) dibangun tiga model: kernel linier, polynomial, dan RBF. Semua memakai parameter default `SVC` (C=1, gamma='scale', degree=3).



In [6]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

splits = {'70:30': 0.30, '80:20': 0.20}
kernels = ['linear', 'poly', 'rbf']

hasil_a = []
for nama_split, test_size in splits.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42)

    for k in kernels:
        model = SVC(kernel=k)
        model.fit(X_train, y_train)
        acc = accuracy_score(y_test, model.predict(X_test))
        hasil_a.append({'Split': nama_split, 'Kernel': k, 'Akurasi': round(acc, 4)})

hasil_a = pd.DataFrame(hasil_a)
print(hasil_a.to_string(index=False))

Split Kernel  Akurasi
70:30 linear   0.9264
70:30   poly   0.4900
70:30    rbf   0.6625
80:20 linear   0.9416
80:20   poly   0.4826
80:20    rbf   0.6562


Percobaan B — dengan standardisasi fitur (`StandardScaler`, di-fit hanya pada data latih). SVM sangat sensitif terhadap skala fitur, jadi percobaan ini dilakukan sebagai perbandingan.

In [7]:
from sklearn.preprocessing import StandardScaler

hasil_b = []
for nama_split, test_size in splits.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42)

    scaler = StandardScaler().fit(X_train)
    X_train_s = scaler.transform(X_train)
    X_test_s = scaler.transform(X_test)

    for k in kernels:
        model = SVC(kernel=k)
        model.fit(X_train_s, y_train)
        acc = accuracy_score(y_test, model.predict(X_test_s))
        hasil_b.append({'Split': nama_split, 'Kernel': k, 'Akurasi': round(acc, 4)})

hasil_b = pd.DataFrame(hasil_b)
print(hasil_b.to_string(index=False))

Split Kernel  Akurasi
70:30 linear   0.9706
70:30   poly   0.9579
70:30    rbf   0.9811
80:20 linear   0.9763
80:20   poly   0.9716
80:20    rbf   0.9826


### Langkah 4 - Tabulasi Akurasi (baris = kernel, kolom = split)

In [8]:
print('Tanpa standardisasi')
print(hasil_a.pivot(index='Kernel', columns='Split', values='Akurasi'))
print()
print('Dengan standardisasi (StandardScaler)')
print(hasil_b.pivot(index='Kernel', columns='Split', values='Akurasi'))

Tanpa standardisasi
Split    70:30   80:20
Kernel                
linear  0.9264  0.9416
poly    0.4900  0.4826
rbf     0.6625  0.6562

Dengan standardisasi (StandardScaler)
Split    70:30   80:20
Kernel                
linear  0.9706  0.9763
poly    0.9579  0.9716
rbf     0.9811  0.9826


## Jawaban Soal 2 — Klasifikasi Siang/Malam dengan Fitur Histogram (SVM RBF, 80:20)


### Langkah 1 - Load Data
Folder `images` disalin dari Drive ke `/content`. Data training dan testing dari Lab 5 digabung, lalu di-split ulang dengan rasio 80:20.

In [9]:
import os, shutil
from pathlib import Path
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import cv2
import numpy as np

if not os.path.exists('/content/images/training'):
    shutil.copytree(BASE + '/images', '/content/images')

%cd /content
train_dir = "images/training/"
test_dir = "images/test/"

def load_dataset(img_dir):
    p = Path(img_dir)
    dirs = p.glob('*')

    img_list = []

    for dir in dirs:
        label = dir.name
        for file in dir.glob('*.jpg'):
            img = mpimg.imread(file)

            if not img is None:
                img_list.append((img, label))

    return img_list

data = load_dataset(train_dir) + load_dataset(test_dir)
print('Jumlah gambar:', len(data))
assert len(data) == 400, "Gambar tidak terbaca, cek isi folder images di Drive"

/content
Jumlah gambar: 400


### Langkah 2 - Ekstraksi Fitur Histogram

Fitur berupa histogram warna ruang HSV (Hue, Saturation, Value) dengan 32 bin per channel, masing-masing dinormalisasi. Hasilnya 96 fitur per gambar. Histogram channel V menggambarkan sebaran kecerahan, yang membedakan siang dan malam.

In [10]:
def hist_feature(image, bins=32):
    image = cv2.resize(image, (1100, 600))
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

    feats = []
    for ch, rng in zip(range(3), ([0, 180], [0, 256], [0, 256])):
        h = cv2.calcHist([hsv], [ch], None, [bins], rng).flatten()
        feats.append(h / h.sum())   # normalisasi

    return np.concatenate(feats)

X = np.array([hist_feature(img) for img, _ in data])
y = np.array([1 if label == 'day' else 0 for _, label in data])   # day = 1, night = 0
print(X.shape, y.shape)

(400, 96) (400,)


### Langkah 3 - Split Data 80:20

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

(320, 96) (80, 96)


### Langkah 4 - Model SVM RBF (default)

In [12]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

model = SVC(kernel='rbf')
model.fit(X_train, y_train)

print('Akurasi train:', accuracy_score(y_train, model.predict(X_train)))
print('Akurasi test :', accuracy_score(y_test, model.predict(X_test)))

Akurasi train: 1.0
Akurasi test : 1.0


### Langkah 5 - Hyperparameter Tuning (C dan gamma) dengan GridSearchCV

In [13]:
from sklearn.model_selection import GridSearchCV

param_grid = {'C': [0.1, 1, 10, 100, 1000],
              'gamma': ['scale', 1, 0.1, 0.01, 0.001]}

grid = GridSearchCV(SVC(kernel='rbf'), param_grid, cv=5, n_jobs=-1)
grid.fit(X_train, y_train)

print('Parameter terbaik  :', grid.best_params_)
print('Akurasi CV terbaik :', grid.best_score_)

Parameter terbaik  : {'C': 10, 'gamma': 'scale'}
Akurasi CV terbaik : 1.0


### Langkah 6 - Evaluasi Model Terbaik pada Data Uji

In [14]:
from sklearn.metrics import classification_report

best = grid.best_estimator_
y_pred = best.predict(X_test)

print('Akurasi test:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['night', 'day']))

Akurasi test: 1.0
              precision    recall  f1-score   support

       night       1.00      1.00      1.00        40
         day       1.00      1.00      1.00        40

    accuracy                           1.00        80
   macro avg       1.00      1.00      1.00        80
weighted avg       1.00      1.00      1.00        80



## Simpan Hasil ke Google Drive

Sel ini menyimpan tabel akurasi Soal 1 ke folder `JS06` di Drive Anda.

In [15]:
hasil_a.to_csv(BASE + '/hasil_soal1_tanpa_standardisasi.csv', index=False)
hasil_b.to_csv(BASE + '/hasil_soal1_dengan_standardisasi.csv', index=False)
print('Tersimpan di', BASE)

Tersimpan di /content/drive/MyDrive/MeachineLearning
